In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import sys

sns.set_theme(style="whitegrid", palette="pastel")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Data Preprocessing

Read the CSV files again to clear any modifications in EDA phase.

In [ ]:
awards_players_df = pd.read_csv('../datasets/awards_players.csv')
coaches_df        = pd.read_csv('../datasets/coaches.csv')
players_teams_df  = pd.read_csv('../datasets/players_teams.csv')
players_df        = pd.read_csv('../datasets/players.csv')
series_post_df    = pd.read_csv('../datasets/series_post.csv')
teams_post_df     = pd.read_csv('../datasets/teams_post.csv')
teams_df          = pd.read_csv('../datasets/teams.csv')

## Data Cleaning

The following variables are redundant or irrelevant, therefore they can be safely removed.

In [ ]:
coaches_df.drop(columns=['lgID'], inplace=True)
awards_players_df.drop(columns=['lgID'], inplace=True)
players_teams_df.drop(columns=['lgID'], inplace=True)
players_df.drop(columns=['firstseason', 'lastseason', 'deathDate'], inplace=True)
series_post_df.drop(columns=['lgIDWinner', 'lgIDLoser'], inplace=True)
teams_post_df.drop(columns=['lgID'], inplace=True)
teams_df.drop(columns=['lgID', 'divID', 'seeded', 'tmORB', 'tmDRB', 'tmTRB', 'opptmORB', 'opptmDRB', 'opptmTRB'], inplace=True)

Before cleaning, identify and remove player entries with no actual game data.

In [ ]:
print("=== IDENTIFYING PLAYERS WITH NO GAME DATA ===\n")

# Find players with minimal/no biographical data
minimal_bio_players = players_df[
    (players_df['height'] == 0) & 
    (players_df['weight'] == 0) & 
    (players_df['pos'].isna()) &
    (players_df['birthDate'] == '0000-00-00')
]

print(f"Players with ALL bio data missing: {len(minimal_bio_players)}")
print(f"Percentage of dataset: {len(minimal_bio_players)/len(players_df)*100:.1f}%")

if len(minimal_bio_players) > 0:
    print("\nSample entries:")
    display(minimal_bio_players[['bioID', 'pos', 'height', 'weight', 'college', 'birthDate']].head())
    
    # Check if any of these players have game statistics
    minimal_ids = set(minimal_bio_players['bioID'].values)
    players_with_stats = players_teams_df[players_teams_df['playerID'].isin(minimal_ids)]
    
    print(f"\nOf these, players appearing in players_teams: {players_with_stats['playerID'].nunique()}")
    
    if len(players_with_stats) == 0:
        print("\n✓ CONCLUSION: These players have NO game statistics")
        print("  → They are 'ghost entries' that never actually played")
        print("  → Safe to remove from dataset")
        
        # Remove these players
        players_df = players_df[~players_df['bioID'].isin(minimal_ids)]
        print(f"\n✓ Removed {len(minimal_bio_players)} players with no game data")
        print(f"  New players_df size: {len(players_df)}")
    else:
        print(f"\nWARNING: {len(players_with_stats)} of these players DO have game stats!")
        print("  → Need to investigate and handle differently")
        display(players_with_stats[['playerID', 'year', 'GP', 'points', 'rebounds']].head())
else:
    print("\n✓ No players with completely missing bio data found")

We identified **78 players (8.7% of the dataset)** with completely missing biographical data:
- height = 0
- weight = 0  
- position = null
- birthDate = '0000-00-00'

After cross-referencing with `players_teams_df`, **NONE of these 78 players have any game statistics**. They are "ghost entries" - players listed in the database who never actually played in any games.

Therefore, these entries are **removed** from the dataset because:
1. They have no analytical value (no games played = no performance data)
2. They would waste computational resources during ML-based imputation
3. Imputing values for players who never played would create misleading synthetic data
4. They cannot be used in any meaningful analysis or modeling

After removal, the dataset is reduced from **893 to 815 players** - all of whom have actual game participation or at least some verifiable biographical information.

### Players Dataset

First, handle some obvious outliers that might skew with missing data training.

In [ ]:
sys.path.append('../logic')

In [ ]:
#from outliers import handle_outliers

#players_df = handle_outliers(players_df)

#### Advanced Missing Data Imputation Strategy

Instead of simple mean/mode imputation, we employ **machine learning-based imputation** with proper experimental design:

**Methodology:**
1. **Position Imputation (Classification)**: Use Random Forest Classifier to predict missing positions based on height and weight
   - Features: height, weight
   - Target: position
   - Handles class imbalance with balanced class weights
   
2. **Height Imputation (Regression)**: Use Random Forest Regressor to predict missing heights based on position and weight
   - Features: position (one-hot encoded), weight
   - Target: height
   
3. **Weight Imputation (Regression)**: Use Random Forest Regressor to predict missing weights based on position and height
   - Features: position (one-hot encoded), height
   - Target: weight

**Experimental Setup (Avoiding Data Leakage):**
- Train-test split (80/20) on complete data for model evaluation
- Models trained only on non-missing observations
- Performance metrics reported on held-out test set
- Final predictions made after retraining on full available data

**Advantages over simple imputation:**
- Captures complex relationships between features
- Provides uncertainty estimates
- Maintains data distribution better than mean/mode
- Can handle non-linear patterns

In [ ]:
from missing_data import impute_missing_data

players_df = impute_missing_data(players_df)

#### Outliers

**Outlier Detection and Analysis**

We perform outlier detection using statistical methods to identify anomalous player measurements:

**Strategy:**
1. **Z-Score Method**: Identify outliers using standardized scores (|z| > 3)
   - Applied to height and weight measurements
   - Flags extreme deviations from the mean

2. **IQR Method**: Use Interquartile Range for robust outlier detection
   - Q1 - 1.5 × IQR and Q3 + 1.5 × IQR
   - Less sensitive to extreme values

3. **Visualization**: Box plots and scatter plots to visually inspect outliers

**Handling Approach:**
- Document outliers for transparency
- Evaluate if outliers are data errors or legitimate extreme values
- Retain legitimate outliers that represent real player measurements
- Flag suspicious values for further investigation

In [ ]:
from outliers import detect_outliers_zscore, detect_outliers_iqr, visualize_outliers

# Detect outliers using Z-Score method
print("=== Z-SCORE OUTLIER DETECTION ===\n")
zscore_outliers = detect_outliers_zscore(players_df)

# Detect outliers using IQR method
print("\n=== IQR OUTLIER DETECTION ===\n")
iqr_outliers = detect_outliers_iqr(players_df)

# Visualize outliers
print("\n=== OUTLIER VISUALIZATIONS ===\n")
visualize_outliers(players_df)

**Results Analysis:**

Both detection methods identified a small percentage of outliers (~1.1-1.3% of the dataset):
- **Z-Score**: 10 outliers with heights 73-80" and weights 239-254 lbs
- **IQR**: 12 outliers, primarily weight-based (>235 lbs), with no height outliers

Key observations:
- All outliers are **Centers (C)** or **Forward-Centers (F-C)** - positions that naturally have larger physiques
- Height range (73-80") is within realistic WNBA bounds (e.g., Margo Dydek was 7'2" / 86")
- Weight range (236-254 lbs) represents heavier players but remains physiologically plausible
- Outliers cluster together in the scatter plot, showing consistent height-weight relationships

**Conclusion:**

**No outlier removal is required.** These are legitimate extreme values representing the natural variation in elite basketball players, not data errors. The detected outliers are:
1. Position-appropriate (Centers are expected to be taller/heavier)
2. Within realistic bounds for professional women's basketball
3. Consistent with each other (no isolated anomalies)

Removing them would eliminate valid information about player diversity and could bias models against larger athletes. The outliers will be **retained** for all subsequent analyses.

### Teams Dataset

#### Outliers

**Outlier Detection and Analysis - Teams Dataset**

We perform outlier detection on team statistics to identify anomalous performance metrics:

**Strategy:**
1. **Z-Score Method**: Identify outliers using standardized scores (|z| > 3)
   - Applied to all numeric team statistics
   - Flags extreme deviations across performance metrics

2. **IQR Method**: Use Interquartile Range for robust outlier detection
   - Q1 - 1.5 × IQR and Q3 + 1.5 × IQR
   - Less sensitive to extreme values

3. **Visualization**: Box plots and scatter plots to visually inspect outliers

**Handling Approach:**
- Document outliers for transparency
- Evaluate if outliers represent exceptional team performance or data errors
- Retain legitimate outliers that represent real historical team performance
- Flag suspicious values for further investigation

In [ ]:
from outliers import detect_outliers_zscore_teams, detect_outliers_iqr_teams, visualize_outliers_teams

# Detect outliers using Z-Score method
print("=== Z-SCORE OUTLIER DETECTION - TEAMS ===\n")
zscore_outliers_teams = detect_outliers_zscore_teams(teams_df)

# Detect outliers using IQR method
print("\n=== IQR OUTLIER DETECTION - TEAMS ===\n")
iqr_outliers_teams = detect_outliers_iqr_teams(teams_df)

# Visualize outliers
print("\n=== OUTLIER VISUALIZATIONS - TEAMS ===\n")
visualize_outliers_teams(teams_df)

**Results Analysis:**

The outlier detection methods identify teams with exceptional or unusual performance metrics across various statistics (attendance, points scored/allowed, rebounds, etc.).

**Key observations:**
- Outliers represent teams with extreme performance (very successful or struggling teams)
- High attendance outliers may represent popular teams or special seasons
- Statistical outliers in scoring/defense metrics reflect exceptional team performance
- Historical context matters - early WNBA seasons may have different baselines

**Conclusion:**

**No outlier removal is required.** The detected outliers represent legitimate historical team performance and should be retained for analysis:
1. Extreme values reflect real team achievements or challenges
2. Outliers provide valuable insights into exceptional seasons
3. Removing them would eliminate important historical variance
4. They are essential for understanding the full spectrum of team performance

The outliers will be **retained** for all subsequent analyses to preserve the complete historical record.

### Players Teams Dataset

Following the same comprehensive cleaning approach used for players and teams datasets.

In [ ]:
# Check current state of players_teams_df
print("=== PLAYERS_TEAMS DATASET - PRE-CLEANING STATE ===\n")
print(f"Shape: {players_teams_df.shape}")
print(f"\nColumns: {players_teams_df.columns.tolist()}")
print(f"\nMissing values:\n{players_teams_df.isnull().sum()}")
print(f"\nBasic statistics:")
display(players_teams_df.describe())

#### Data Quality Check: Extreme Outliers

Before imputation, check for unrealistic values in key statistical columns that might indicate data entry errors.

In [ ]:
print("=== PRE-IMPUTATION DATA QUALITY CHECK - PLAYERS_TEAMS ===\n")

# Check for negative values in counting statistics (should be >= 0)
counting_stats = ['GP', 'GS', 'minutes', 'points', 'rebounds', 'assists', 'steals', 'blocks', 
                  'turnovers', 'fgAttempted', 'fgMade', 'ftAttempted', 'ftMade',
                  'PostGP', 'PostGS', 'PostMinutes', 'PostPoints', 'PostRebounds', 'PostAssists']

negative_found = False
for col in counting_stats:
    if col in players_teams_df.columns:
        negative_count = (players_teams_df[col] < 0).sum()
        if negative_count > 0:
            negative_found = True
            print(f"Found {negative_count} negative values in {col}")
            display(players_teams_df[players_teams_df[col] < 0][['playerID', 'year', 'tmID', col]])
            # Convert negative values to 0 for proper imputation
            players_teams_df.loc[players_teams_df[col] < 0, col] = 0
            print(f"✓ Converted to zero for imputation\n")

if not negative_found:
    print("✓ No negative values found in counting statistics\n")

# Check for impossible percentages (made > attempted)
if 'fgMade' in players_teams_df.columns and 'fgAttempted' in players_teams_df.columns:
    impossible_fg = players_teams_df['fgMade'] > players_teams_df['fgAttempted']
    if impossible_fg.sum() > 0:
        print(f"Found {impossible_fg.sum()} records where fgMade > fgAttempted")
        display(players_teams_df[impossible_fg][['playerID', 'year', 'fgAttempted', 'fgMade']])
        print("These may be data quality issues that need investigation\n")

if 'ftMade' in players_teams_df.columns and 'ftAttempted' in players_teams_df.columns:
    impossible_ft = players_teams_df['ftMade'] > players_teams_df['ftAttempted']
    if impossible_ft.sum() > 0:
        print(f"Found {impossible_ft.sum()} records where ftMade > ftAttempted")
        display(players_teams_df[impossible_ft][['playerID', 'year', 'ftAttempted', 'ftMade']])

print("Pre-imputation quality check complete!")

#### Missing Data Imputation

Apply ML-based imputation strategy for missing values in player team statistics.

**Methodology:**
1. **Identify patterns**: Analyze which statistics have missing values and their relationships
2. **Feature engineering**: Use available statistics to predict missing values
3. **ML-based imputation**: Use Random Forest and other methods to predict missing values based on:
   - Player characteristics (from related data)
   - Team performance metrics
   - Temporal patterns (year-based trends)
   - Correlated statistics (e.g., rebounds vs minutes played)

**Advantages:**
- Captures complex relationships between statistics
- Maintains statistical distributions
- Accounts for position-specific and era-specific patterns
- More accurate than simple mean/median imputation

In [ ]:
from missing_data import impute_missing_data_players_teams

players_teams_df = impute_missing_data_players_teams(players_teams_df)

#### Outliers

**Outlier Detection and Analysis - Players Teams Dataset**

We perform outlier detection on player team statistics to identify anomalous performance:

**Strategy:**
1. **Z-Score Method**: Identify outliers using standardized scores (|z| > 3)
   - Applied to key performance statistics (points, rebounds, assists, etc.)
   - Flags extreme deviations from the mean

2. **IQR Method**: Use Interquartile Range for robust outlier detection
   - Q1 - 1.5 × IQR and Q3 + 1.5 × IQR
   - Less sensitive to extreme values

3. **Visualization**: Box plots and scatter plots to visually inspect outliers

**Columns to analyze:**
- Regular season stats: points, rebounds, assists, steals, blocks, minutes
- Efficiency metrics: field goal percentage, free throw percentage
- Playoff statistics (for players who participated)

**Handling Approach:**
- Document outliers for transparency
- Evaluate if outliers represent exceptional performances or data errors
- Retain legitimate outliers that represent real player achievements
- Flag suspicious values for further investigation

In [ ]:
from outliers import detect_outliers_zscore_players_teams, detect_outliers_iqr_players_teams, visualize_outliers_players_teams

# Detect outliers using Z-Score method
print("=== Z-SCORE OUTLIER DETECTION - PLAYERS_TEAMS ===\n")
zscore_outliers_pt = detect_outliers_zscore_players_teams(players_teams_df)

# Detect outliers using IQR method
print("\n=== IQR OUTLIER DETECTION - PLAYERS_TEAMS ===\n")
iqr_outliers_pt = detect_outliers_iqr_players_teams(players_teams_df)

# Visualize outliers
print("\n=== OUTLIER VISUALIZATIONS - PLAYERS_TEAMS ===\n")
visualize_outliers_players_teams(players_teams_df)

**Results Analysis:**

The outlier detection methods identify player-team season performances with exceptional statistics across various metrics (points, rebounds, assists, steals, blocks, minutes).

**Key observations:**
- Outliers represent players with exceptional season performances (All-Stars, MVP candidates)
- High values in counting stats (points, rebounds, assists) often correlate with high minutes played
- Statistical outliers reflect dominant individual performances
- Some outliers may represent partial season data or unique circumstances

**Conclusion:**

**No outlier removal is required.** The detected outliers represent legitimate exceptional player performances and should be retained for analysis:
1. Extreme values reflect real player achievements and elite performances
2. Outliers provide valuable insights into what constitutes exceptional play
3. Removing them would eliminate important information about star players
4. They are essential for understanding the full spectrum of player performance
5. Many outliers likely represent award-winning seasons or Hall of Fame performances

The outliers will be **retained** for all subsequent analyses to preserve the complete performance record and enable proper analysis of exceptional talent.

### Feature Engineering

In [ ]:
# Total wins and losses (regular season + playoff)
coaches_df["total_w"] = coaches_df["won"] + coaches_df["post_wins"]
coaches_df["total_l"] = coaches_df["lost"] + coaches_df["post_losses"]

In [ ]:
# per-game averages for key stats
players_df['ppg'] = players_teams_df['points'] / players_teams_df['GP']
players_df['apg'] = players_teams_df['assists'] / players_teams_df['GP']
players_df['rpg'] = players_teams_df['rebounds'] / players_teams_df['GP']
players_df['spg'] = players_teams_df['steals'] / players_teams_df['GP']
players_df['bpg'] = players_teams_df['blocks'] / players_teams_df['GP']
players_df['fg_pct'] = players_teams_df['fgMade'] / players_teams_df['fgAttempted']

In [ ]:
# Teams' win percentage
teams_df['win_pct'] = (teams_df['won'] / teams_df['GP']) * 100

In [ ]:
# Assist to turnover Ratio
players_teams_df["ast_to"] = (players_teams_df["assists"] + players_teams_df["PostAssists"]) / (players_teams_df["turnovers"] + players_teams_df["PostTurnovers"])

In [ ]:
## Player Efficiency Rating (PER)

players_teams_df["NBA_PER"] = (players_teams_df['points'] + players_teams_df['rebounds']+ players_teams_df['assists'] + players_teams_df['steals'] + players_teams_df['blocks']
  - ((players_teams_df['fgAttempted'] - players_teams_df['fgMade']) + (players_teams_df['ftAttempted'] - players_teams_df['ftMade']) + players_teams_df['turnovers'])
) / players_teams_df["GP"]


In [ ]:
# True Shooting Percentage - how efficiently a player converts their scoring opportunities
players_teams_df["ts_pct"] = players_teams_df['points'] / (2 * (players_teams_df["fgAttempted"] + (0.44 * players_teams_df["ftAttempted"])))

### Data Transformation

In [ ]:
# Teams Stats Histograms
teams_numeric_cols = teams_df.select_dtypes(include=['float64', 'int64']).filter(regex=r'^(d_|o_)')

teams_numeric_cols.describe()

In [ ]:
plt.figure(figsize=(16, 12))

for i, col in enumerate(teams_numeric_cols):
    plt.subplot(6, 6, i+1)
    sns.histplot(teams_numeric_cols[col], bins=30, kde=True, color='skyblue')
    plt.title(col)
    plt.tight_layout()

In [ ]:
# log transform
teams_numeric_cols_log = np.log(teams_df[teams_numeric_cols.columns])

In [ ]:
plt.figure(figsize=(16, 12))

for i, col in enumerate(teams_numeric_cols_log):
    plt.subplot(6, 6, i+1)
    sns.histplot(teams_numeric_cols_log[col], bins=30, kde=True, color='skyblue')
    plt.title(col)
    plt.tight_layout()

In [ ]:
# Convert Y/N to 0/1
teams_df['playoff'] = teams_df.playoff.map(dict(Y=1, N=0))

In [ ]:
# Isolation Forest
#iso = IsolationForest(contamination=0.01, random_state=42) # random state para ter smp mm res
#y_pred = iso.fit_predict(teams_df.select_dtypes(include=['number']))
#teams_df_outliers = teams_df[y_pred == -1]

#display(teams_df_outliers)

## Data Integration

Combining data from multiple sources to create comprehensive, unified datasets for analysis.

#### 1. Verify Data Formats and Key Columns

Before integration, we need to check the structure of each dataset and identify common keys.

In [ ]:
# Check key columns in each dataset for integration
print("=== Dataset Structures for Integration ===\n")

print("players_df columns:", players_df.columns.tolist()[:10], "...")
print("  Key: playerID")
print("  Rows:", len(players_df))

print("\nplayers_teams_df columns:", players_teams_df.columns.tolist()[:10], "...")
print("  Keys: playerID, year, tmID")
print("  Rows:", len(players_teams_df))

print("\nawards_players_df columns:", awards_players_df.columns.tolist())
print("  Keys: playerID, year")
print("  Rows:", len(awards_players_df))

print("\nteams_df columns:", teams_df.columns.tolist()[:10], "...")
print("  Keys: year, tmID")
print("  Rows:", len(teams_df))

print("\nteams_post_df columns:", teams_post_df.columns.tolist())
print("  Keys: year, tmID")
print("  Rows:", len(teams_post_df))

print("\ncoaches_df columns:", coaches_df.columns.tolist())
print("  Keys: coachID, year, tmID")
print("  Rows:", len(coaches_df))

print("\nseries_post_df columns:", series_post_df.columns.tolist())
print("  Keys: year, round, tmIDWinner, tmIDLoser")
print("  Rows:", len(series_post_df))

#### 2. Standardize Data Formats

Ensure consistent data types and formats across datasets for successful integration.

In [ ]:
# Standardize year formats (ensure all are integers)
for df_name, df in [('players_teams_df', players_teams_df), 
                     ('awards_players_df', awards_players_df),
                     ('teams_df', teams_df), 
                     ('teams_post_df', teams_post_df),
                     ('coaches_df', coaches_df),
                     ('series_post_df', series_post_df)]:
    if 'year' in df.columns:
        df['year'] = df['year'].astype(int)
        print(f"{df_name}: year converted to int")

# Standardize team IDs (ensure strings and strip whitespace)
for df_name, df in [('players_teams_df', players_teams_df),
                     ('teams_df', teams_df),
                     ('teams_post_df', teams_post_df),
                     ('coaches_df', coaches_df)]:
    if 'tmID' in df.columns:
        df['tmID'] = df['tmID'].astype(str).str.strip()
        print(f"{df_name}: tmID standardized")

# Standardize team IDs in series_post_df
series_post_df['tmIDWinner'] = series_post_df['tmIDWinner'].astype(str).str.strip()
series_post_df['tmIDLoser'] = series_post_df['tmIDLoser'].astype(str).str.strip()
print("series_post_df: tmIDWinner and tmIDLoser standardized")

# Standardize player IDs
for df_name, df in [('players_teams_df', players_teams_df),
                     ('awards_players_df', awards_players_df)]:
    if 'playerID' in df.columns:
        df['playerID'] = df['playerID'].astype(str).str.strip()
        print(f"{df_name}: playerID standardized")

# Standardize bioID in players_df
if 'bioID' in players_df.columns:
    players_df['bioID'] = players_df['bioID'].astype(str).str.strip()
    print("players_df: bioID standardized")

# Standardize coach IDs
coaches_df['coachID'] = coaches_df['coachID'].astype(str).str.strip()
print("coaches_df: coachID standardized")

print("\n=== Data Format Standardization Complete ===")

#### 3. Create Integrated Player Dataset

Merge player information with their team performance and awards.

In [ ]:
# Merge players with their team statistics
# Note: players_df uses 'bioID' while players_teams_df uses 'playerID'
# We'll join players_teams with basic player attributes

players_integrated = players_teams_df.copy()

print(f"Starting with players_teams data: {len(players_integrated)} records")
print(f"  Columns: {players_integrated.shape[1]}")

# Add awards information (count awards per player per year)
awards_summary = awards_players_df.groupby(['playerID', 'year']).agg({
    'award': 'count'
}).rename(columns={'award': 'num_awards'}).reset_index()

players_integrated = players_integrated.merge(
    awards_summary,
    on=['playerID', 'year'],
    how='left'
)

# Fill NaN awards with 0 (players with no awards)
players_integrated['num_awards'] = players_integrated['num_awards'].fillna(0).astype(int)

print(f"Added awards information")
print(f"  Players with awards in dataset: {(players_integrated['num_awards'] > 0).sum()}")

# Add team performance data
team_performance_cols = ['year', 'tmID', 'playoff', 'won', 'lost', 'confID', 'GP', 'name']
players_integrated = players_integrated.merge(
    teams_df[team_performance_cols],
    on=['year', 'tmID'],
    how='left',
    suffixes=('', '_team')
)

print(f"Added team performance data")

# Display sample
print("\n=== Integrated Player Dataset Sample ===")
display(players_integrated.head())
print(f"\nTotal records: {len(players_integrated)}")
print(f"Total columns: {players_integrated.shape[1]}")

#### 4. Create Integrated Team Dataset

Merge team regular season data with playoff performance and coaching information.

In [ ]:
# Start with teams regular season data
teams_integrated = teams_df.copy()

print(f"Starting with teams_df: {len(teams_integrated)} records")

# Add playoff performance
playoff_cols = ['year', 'tmID', 'W', 'L']
teams_integrated = teams_integrated.merge(
    teams_post_df[playoff_cols],
    on=['year', 'tmID'],
    how='left',
    suffixes=('', '_playoff')
)

# Rename playoff columns for clarity
teams_integrated.rename(columns={'W': 'playoff_wins', 'L': 'playoff_losses'}, inplace=True)

print(f"Added playoff data")
print(f"  Teams with playoff appearances: {teams_integrated['playoff_wins'].notna().sum()}")

# Add coach information (aggregate if multiple coaches per team-year)
coaches_agg = coaches_df.groupby(['year', 'tmID']).agg({
    'coachID': lambda x: ', '.join(x),  # Combine multiple coaches
    'won': 'sum',
    'lost': 'sum',
    'post_wins': 'sum',
    'post_losses': 'sum'
}).reset_index()

coaches_agg.rename(columns={
    'coachID': 'coaches',
    'won': 'coach_won',
    'lost': 'coach_lost',
    'post_wins': 'coach_post_wins',
    'post_losses': 'coach_post_losses'
}, inplace=True)

teams_integrated = teams_integrated.merge(
    coaches_agg,
    on=['year', 'tmID'],
    how='left'
)

print(f"Added coaching data")
print(f"  Teams with coach info: {teams_integrated['coaches'].notna().sum()}")

# Add championship information from series_post
champions = series_post_df[series_post_df['round'] == 'F'][['year', 'tmIDWinner']].copy()
champions['champion'] = 1
champions.rename(columns={'tmIDWinner': 'tmID'}, inplace=True)

teams_integrated = teams_integrated.merge(
    champions,
    on=['year', 'tmID'],
    how='left'
)

teams_integrated['champion'] = teams_integrated['champion'].fillna(0).astype(int)

print(f"Added championship flag")
print(f"  Championship teams: {teams_integrated['champion'].sum()}")

# Display sample
print("\n=== Integrated Team Dataset Sample ===")
display(teams_integrated.head())
print(f"\nTotal records: {len(teams_integrated)}")
print(f"Total columns: {teams_integrated.shape[1]}")

#### 5. Create Coach Performance Dataset

Integrate coach data with team performance to analyze coaching effectiveness.

In [ ]:
# Merge coaches with team performance
coaches_integrated = coaches_df.merge(
    teams_df[['year', 'tmID', 'name', 'confID', 'playoff', 'GP', 'arena', 'attend']],
    on=['year', 'tmID'],
    how='left'
)

print(f"Merged coaches with team info: {len(coaches_integrated)} records")

# Add playoff results for coaches
coaches_integrated = coaches_integrated.merge(
    teams_post_df[['year', 'tmID', 'W', 'L']],
    on=['year', 'tmID'],
    how='left',
    suffixes=('', '_team_playoff')
)

coaches_integrated.rename(columns={'W': 'team_playoff_W', 'L': 'team_playoff_L'}, inplace=True)

print(f"Added team playoff results")

# Add championship flag
champions = series_post_df[series_post_df['round'] == 'F'][['year', 'tmIDWinner']].copy()
champions['champion'] = 1
champions.rename(columns={'tmIDWinner': 'tmID'}, inplace=True)

coaches_integrated = coaches_integrated.merge(
    champions[['year', 'tmID', 'champion']],
    on=['year', 'tmID'],
    how='left'
)

coaches_integrated['champion'] = coaches_integrated['champion'].fillna(0).astype(int)

print(f"Added championship information")
print(f"  Coaches who won championships: {coaches_integrated['champion'].sum()}")

# Calculate win percentage
coaches_integrated['win_pct'] = coaches_integrated['won'] / (coaches_integrated['won'] + coaches_integrated['lost']) * 100

# Display sample
print("\n=== Integrated Coach Dataset Sample ===")
display(coaches_integrated.head())
print(f"\nTotal records: {len(coaches_integrated)}")
print(f"Total columns: {coaches_integrated.shape[1]}")

#### 6. Create Award Winners Dataset

Integrate award information with player and team data for comprehensive analysis.

In [ ]:
# Merge awards with player statistics
awards_integrated = awards_players_df.copy()

print(f"Starting with awards data: {len(awards_integrated)} records")

# Add player statistics for the award year
player_stats_cols = ['playerID', 'year', 'tmID', 'GP', 'points', 'rebounds', 'assists', 
                     'steals', 'blocks', 'turnovers']
awards_integrated = awards_integrated.merge(
    players_teams_df[player_stats_cols],
    on=['playerID', 'year'],
    how='left'
)

print(f"Added player statistics for award years")

# Add team information
team_info_cols = ['year', 'tmID', 'name', 'playoff', 'won', 'lost']
awards_integrated = awards_integrated.merge(
    teams_df[team_info_cols],
    on=['year', 'tmID'],
    how='left'
)

print(f"Added team information")

# Add championship flag
awards_integrated = awards_integrated.merge(
    champions[['year', 'tmID', 'champion']],
    on=['year', 'tmID'],
    how='left'
)

awards_integrated['champion'] = awards_integrated['champion'].fillna(0).astype(int)

print(f"Added championship information")
print(f"  Awards won by players on championship teams: {awards_integrated['champion'].sum()}")

# Display sample
print("\n=== Integrated Awards Dataset Sample ===")
display(awards_integrated.head(10))
print(f"\nTotal records: {len(awards_integrated)}")
print(f"Total columns: {awards_integrated.shape[1]}")
print(f"\nAward distribution:")
display(awards_integrated['award'].value_counts())

#### 7. Validate Integration Quality

Check for data quality issues after integration.

In [ ]:
print("=== Data Integration Quality Report ===\n")

# Check players_integrated
print("1. PLAYERS INTEGRATED DATASET")
print(f"   Total records: {len(players_integrated)}")
print(f"   Missing values by column:")
missing_players = players_integrated.isnull().sum()
display(missing_players[missing_players > 0])
print(f"   Duplicate records: {players_integrated.duplicated().sum()}")

# Check teams_integrated
print("\n2. TEAMS INTEGRATED DATASET")
print(f"   Total records: {len(teams_integrated)}")
print(f"   Missing values by column:")
missing_teams = teams_integrated.isnull().sum()
display(missing_teams[missing_teams > 0])
print(f"   Duplicate records: {teams_integrated.duplicated().sum()}")
print(f"   Teams with playoff data: {teams_integrated['playoff_wins'].notna().sum()}")

# Check coaches_integrated
print("\n3. COACHES INTEGRATED DATASET")
print(f"   Total records: {len(coaches_integrated)}")
print(f"   Missing values by column:")
missing_coaches = coaches_integrated.isnull().sum()
display(missing_coaches[missing_coaches > 0])
print(f"   Duplicate records: {coaches_integrated.duplicated().sum()}")

# Check awards_integrated
print("\n4. AWARDS INTEGRATED DATASET")
print(f"   Total records: {len(awards_integrated)}")
print(f"   Missing values by column:")
missing_awards = awards_integrated.isnull().sum()
display(missing_awards[missing_awards > 0])
print(f"   Duplicate records: {awards_integrated.duplicated().sum()}")

# Entity matching verification
print("\n5. ENTITY MATCHING VERIFICATION")
print(f"   Unique players in players_integrated: {players_integrated['playerID'].nunique()}")
print(f"   Unique teams in teams_integrated: {teams_integrated['tmID'].nunique()}")
print(f"   Unique coaches in coaches_integrated: {coaches_integrated['coachID'].nunique()}")
print(f"   Year range in all datasets: {players_integrated['year'].min()}-{players_integrated['year'].max()}")

print("\nIntegration quality check complete!")

#### 8. Summary of Integrated Datasets

The integrated datasets are now ready for advanced analysis and modeling.

In [ ]:
print("=== INTEGRATED DATASETS SUMMARY ===\n")

datasets_info = {
    'players_integrated': {
        'dataset': players_integrated,
        'description': 'Player stats merged with personal info, awards, and team performance',
        'key_use_cases': [
            'Analyze player performance across teams',
            'Correlate awards with team success',
            'Study player statistics over time'
        ]
    },
    'teams_integrated': {
        'dataset': teams_integrated,
        'description': 'Team regular season data merged with playoff performance, coaching, and championships',
        'key_use_cases': [
            'Analyze team performance trends',
            'Study playoff success factors',
            'Examine coaching impact on team success'
        ]
    },
    'coaches_integrated': {
        'dataset': coaches_integrated,
        'description': 'Coach data merged with team performance and championship results',
        'key_use_cases': [
            'Evaluate coaching effectiveness',
            'Compare coach performance across teams',
            'Study championship-winning coaches'
        ]
    },
    'awards_integrated': {
        'dataset': awards_integrated,
        'description': 'Award winners merged with player stats, team info, and championships',
        'key_use_cases': [
            'Analyze award winner characteristics',
            'Study correlation between awards and championships',
            'Compare performance of award winners'
        ]
    }
}

for name, info in datasets_info.items():
    print(f"{name.upper()}")
    print(f"   Records: {len(info['dataset']):,}")
    print(f"   Columns: {info['dataset'].shape[1]}")
    print(f"   Description: {info['description']}")
    #print(f"   Key Use Cases:")
    for use_case in info['key_use_cases']:
        print(f"      • {use_case}")
    print()

print("\nAll datasets have been successfully integrated and are ready for analysis and modeling.")